In [1]:
from tokenizer.tokenizer import Tokenizer

tokenizer = Tokenizer()

tokenizer.train(
    "hello hello hello hello",
    vocab_size=270
)

encoded = tokenizer.encode("hello hello")

print(encoded)
print(tokenizer.decode(encoded))

[260, 259]
hello hello


In [1]:
from tokenizer.tokenizer import Tokenizer

tokenizer = Tokenizer()

tokenizer.train(
    "hello hello hello hello",
    vocab_size=270,
)

tokenizer.save("../results/hello_tokenizer.json")

In [2]:
loaded = Tokenizer()
loaded.load("../results/hello_tokenizer.json")

encoded = loaded.encode("hello hello")

print(encoded)
print(loaded.decode(encoded))

[260, 259]
hello hello


# Unicode and Byte-Level Representation

In [1]:
from tokenizer.unicode_utils import text_to_bytes

texts = {
    "English": "hello",
    "Nepali": "नेपाल",
    "Hindi": "भारत",
}

for language, text in texts.items():
    ids = text_to_bytes(text)

    print(f"{language}: {text}")
    print(f"Characters: {len(text)}")
    print(f"UTF-8 bytes: {len(ids)}")
    print(f"Bytes: {ids}")
    print()

English: hello
Characters: 5
UTF-8 bytes: 5
Bytes: [104, 101, 108, 108, 111]

Nepali: नेपाल
Characters: 5
UTF-8 bytes: 15
Bytes: [224, 164, 168, 224, 165, 135, 224, 164, 170, 224, 164, 190, 224, 164, 178]

Hindi: भारत
Characters: 4
UTF-8 bytes: 12
Bytes: [224, 164, 173, 224, 164, 190, 224, 164, 176, 224, 164, 164]



In [2]:
for language, text in texts.items():
    ids = text_to_bytes(text)

    print(
        language,
        "bytes/character =",
        len(ids) / len(text)
    )

English bytes/character = 1.0
Nepali bytes/character = 3.0
Hindi bytes/character = 3.0


# Baseline BPE Comparison

In [3]:
from tokenizer.tokenizer import Tokenizer

texts = {
    "English": """
    machine learning is a method of artificial intelligence.
    machine learning models learn patterns from data.
    """,

    "Hindi": """
    मशीन लर्निंग कृत्रिम बुद्धिमत्ता की एक विधि है।
    मशीन लर्निंग मॉडल डेटा से पैटर्न सीखते हैं।
    """,

    "Nepali": """
    मेसिन लर्निङ कृत्रिम बुद्धिमत्ताको एउटा विधि हो।
    मेसिन लर्निङ मोडेलहरूले डाटाबाट ढाँचाहरू सिक्छन्।
    """
}

In [4]:
tokenizers = {}

for language, text in texts.items():
    tokenizer = Tokenizer()

    tokenizer.train(
        text,
        vocab_size=300
    )

    tokenizers[language] = tokenizer

    print(
        language,
        "vocab size:",
        tokenizer.vocab_size,
        "merges:",
        len(tokenizer.merges)
    )

English vocab size: 300 merges: 44
Hindi vocab size: 300 merges: 44
Nepali vocab size: 300 merges: 44


In [5]:
test_sentences = {
    "English": "machine learning",
    "Hindi": "मशीन लर्निंग",
    "Nepali": "मेसिन लर्निङ",
}

for language, sentence in test_sentences.items():
    tokenizer = tokenizers[language]

    tokens = tokenizer.encode(sentence)

    print(f"\n{language}")
    print("Text:", sentence)
    print("Token IDs:", tokens)
    print("Number of tokens:", len(tokens))


English
Text: machine learning
Token IDs: [109, 97, 99, 104, 257, 101, 264, 257, 103]
Number of tokens: 9

Hindi
Text: मशीन लर्निंग
Token IDs: [256, 174, 256, 182, 267, 268, 178, 270, 168, 263, 130, 256, 151]
Number of tokens: 13

Nepali
Text: मेसिन लर्निङ
Token IDs: [256, 174, 270, 271, 168, 258, 178, 267, 264, 168, 260, 153]
Number of tokens: 12


In [6]:
for language, tokenizer in tokenizers.items():
    print(f"\n===== {language} =====")

    for token_id, token_bytes in tokenizer.vocab.items():
        if token_id >= 256:
            print(
                token_id,
                repr(token_bytes),
                token_bytes.decode("utf-8", errors="replace")
            )


===== English =====
256 b'  '   
257 b'in' in
258 b'ar' ar
259 b'\n  ' 
  
260 b'\n    ' 
    
261 b' l'  l
262 b' le'  le
263 b' lear'  lear
264 b' learn'  learn
265 b'\n    m' 
    m
266 b'\n    ma' 
    ma
267 b'\n    mac' 
    mac
268 b'\n    mach' 
    mach
269 b'\n    machin' 
    machin
270 b'\n    machine' 
    machine
271 b'\n    machine learn' 
    machine learn
272 b'\n    machine learnin' 
    machine learnin
273 b'\n    machine learning' 
    machine learning
274 b'\n    machine learning ' 
    machine learning 
275 b's ' s 
276 b'od' od
277 b'te' te
278 b'at' at
279 b'\n    machine learning i' 
    machine learning i
280 b'\n    machine learning is ' 
    machine learning is 
281 b'\n    machine learning is a' 
    machine learning is a
282 b'\n    machine learning is a ' 
    machine learning is a 
283 b'\n    machine learning is a m' 
    machine learning is a m
284 b'\n    machine learning is a me' 
    machine learning is a me
285 b'\n    machine learning is a met' 


In [7]:
texts = {
    "English": (
        "machine learning is a method of artificial intelligence. "
        "machine learning models learn patterns from data. "
        "machine learning is useful."
    ),

    "Hindi": (
        "मशीन लर्निंग कृत्रिम बुद्धिमत्ता की एक विधि है। "
        "मशीन लर्निंग मॉडल डेटा से पैटर्न सीखते हैं। "
        "मशीन लर्निंग उपयोगी है।"
    ),

    "Nepali": (
        "मेसिन लर्निङ कृत्रिम बुद्धिमत्ताको एउटा विधि हो। "
        "मेसिन लर्निङ मोडेलहरूले डाटाबाट ढाँचाहरू सिक्छन्। "
        "मेसिन लर्निङ उपयोगी छ।"
    )
}

In [8]:
tokenizers = {}

for language, text in texts.items():
    tokenizer = Tokenizer()

    tokenizer.train(
        text,
        vocab_size=300
    )

    tokenizers[language] = tokenizer

    print(
        language,
        "vocab size:",
        tokenizer.vocab_size,
        "merges:",
        len(tokenizer.merges)
    )

English vocab size: 300 merges: 44
Hindi vocab size: 300 merges: 44
Nepali vocab size: 300 merges: 44


In [9]:
test_sentences = {
    "English": "machine learning",
    "Hindi": "मशीन लर्निंग",
    "Nepali": "मेसिन लर्निङ",
}

for language, sentence in test_sentences.items():
    tokenizer = tokenizers[language]

    tokens = tokenizer.encode(sentence)

    print(f"\n{language}")
    print("Text:", sentence)
    print("Token IDs:", tokens)
    print("Number of tokens:", len(tokens))


English
Text: machine learning
Token IDs: [109, 269]
Number of tokens: 2

Hindi
Text: मशीन लर्निंग
Token IDs: [256, 278]
Number of tokens: 2

Nepali
Text: मेसिन लर्निङ
Token IDs: [256, 279]
Number of tokens: 2


In [10]:
for language, tokenizer in tokenizers.items():
    print(f"\n===== {language} =====")

    for token_id, token_bytes in tokenizer.vocab.items():
        if token_id >= 256:
            print(
                token_id,
                repr(token_bytes),
                token_bytes.decode("utf-8", errors="replace")
            )


===== English =====
256 b'in' in
257 b'ar' ar
258 b' l'  l
259 b' le'  le
260 b' lear'  lear
261 b' learn'  learn
262 b' m'  m
263 b'ac' ac
264 b'ach' ach
265 b'achin' achin
266 b'achine' achine
267 b'achine learn' achine learn
268 b'achine learnin' achine learnin
269 b'achine learning' achine learning
270 b's ' s 
271 b'achine learning ' achine learning 
272 b'achine learning i' achine learning i
273 b'achine learning is ' achine learning is 
274 b'od' od
275 b'te' te
276 b'. m' . m
277 b'at' at
278 b'machine learning is ' machine learning is 
279 b'machine learning is a' machine learning is a
280 b'machine learning is a m' machine learning is a m
281 b'machine learning is a me' machine learning is a me
282 b'machine learning is a met' machine learning is a met
283 b'machine learning is a meth' machine learning is a meth
284 b'machine learning is a method' machine learning is a method
285 b'machine learning is a method ' machine learning is a method 
286 b'machine learning is a metho